In [14]:
# Cell 1 - load alerts CSV

import pandas as pd
from pathlib import Path

# === Path to your alerts CSV ===
CSV_PATH = Path(r"C:\Users\ishaa\OneDrive\Desktop\Features-main\Features-main\data\output\vehicle_alerts.csv")

df = pd.read_csv(
    CSV_PATH,
    dtype=str,               # keep everything as string so we don't break encoded stuff
    keep_default_na=False,
    na_values=[""],
    engine="python",
)

print(f"Loaded: {CSV_PATH}")
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("Columns:", list(df.columns))


Loaded: C:\Users\ishaa\OneDrive\Desktop\Features-main\Features-main\data\output\vehicle_alerts.csv
Shape: 110 rows, 14 columns
Columns: ['alert_id', 'vehicle_id', 'module', 'start_ts', 'end_ts', 'duration_s', 'n_points', 'peak_composite', 'mean_composite', 'severity', 'severity_label', 'top_features_b64', 'date', 'row_hash']


In [15]:
# Cell 2 - detect base64 JSON encoded columns

import base64
import json

def looks_like_base64_json(value: str) -> bool:
    """
    Heuristic:
    - ignore empty strings
    - try base64 decode
    - then try json.loads on the decoded bytes
    If both succeed, we call it base64 JSON.
    """
    if value is None:
        return False

    s = str(value).strip()
    if not s:
        return False

    try:
        # base64 decoding
        decoded = base64.b64decode(s, validate=False)
    except Exception:
        return False

    try:
        json.loads(decoded.decode("utf-8"))
        return True
    except Exception:
        return False


encoded_json_cols = []

max_samples_per_col = 50

for col in df.columns:
    # get some non-empty samples from this column
    non_empty = df[col].astype(str).str.strip()
    non_empty = non_empty[non_empty != ""]
    if non_empty.empty:
        continue

    sample = non_empty.sample(
        n=min(max_samples_per_col, len(non_empty)),
        random_state=42
    )

    successes = 0
    total = len(sample)

    for v in sample:
        if looks_like_base64_json(v):
            successes += 1

    if total > 0 and successes / total >= 0.8:
        encoded_json_cols.append(col)
        print(f"Column '{col}' is a base64 JSON ({successes}/{total} samples decoded).")
    else:
        continue




Column 'top_features_b64' is a base64 JSON (50/50 samples decoded).


In [16]:
def decode_base64_json_to_string(value: str) -> str:
    """
    Decode base64 JSON and return a compact JSON string.
    If decoding fails or value is empty, return the original value.
    (You can change this to return "" on failure if you prefer.)
    """
    if value is None:
        return ""

    s = str(value).strip()
    if not s:
        return ""

    try:
        decoded = base64.b64decode(s, validate=False)
        obj = json.loads(decoded.decode("utf-8"))
        # compact one-line JSON string (no extra spaces, Excel-friendly)
        return json.dumps(obj, ensure_ascii=False, separators=(",", ":"))
    except Exception:
        # fall back to original if it's not valid base64 JSON
        return s

df_decoded = df.copy()

for col in encoded_json_cols:
    new_col = f"{col}_decoded_json"
    print(f"Decoding column '{col}' into '{new_col}'...")
    df_decoded[new_col] = df_decoded[col].apply(decode_base64_json_to_string)

print("Decoded columns added:", [f"{c}_decoded_json" for c in encoded_json_cols])


Decoding column 'top_features_b64' into 'top_features_b64_decoded_json'...
Decoded columns added: ['top_features_b64_decoded_json']


In [17]:
out_path = CSV_PATH.with_name(CSV_PATH.stem + "_decoded_json" + CSV_PATH.suffix)

df_decoded.to_csv(
    out_path,
    index=False,
    encoding="utf-8",
    lineterminator="\n"
)

print(f"Wrote decoded CSV to: {out_path}")


Wrote decoded CSV to: C:\Users\ishaa\OneDrive\Desktop\Features-main\Features-main\data\output\vehicle_alerts_decoded_json.csv


In [9]:
# Cell 5 - decode base64 JSON columns for all module CSVs (raw + inference)

import base64
import json
from pathlib import Path
import pandas as pd

RAW_DIR = Path(r"C:\Users\ishaa\OneDrive\Desktop\Features-main\Features-main\data")
OUT_DIR = Path(r"C:\Users\ishaa\OneDrive\Desktop\Features-main\Features-main\data\output")

RAW_FILES = [
    RAW_DIR / "battery.csv",
    RAW_DIR / "body.csv",
    RAW_DIR / "engine.csv",
    RAW_DIR / "transmission.csv",
    RAW_DIR / "tyre.csv",
]

INF_FILES = [
    OUT_DIR / "battery_inference.csv",
    OUT_DIR / "body_inference.csv",
    OUT_DIR / "engine_inference.csv",
    OUT_DIR / "transmission_inference.csv",
    OUT_DIR / "tyre_inference.csv",
]

ALL_FILES = RAW_FILES + INF_FILES


def looks_like_base64_json(val: str) -> bool:
    if val is None:
        return False
    s = str(val).strip()
    if not s:
        return False
    try:
        decoded = base64.b64decode(s, validate=False)
    except Exception:
        return False
    try:
        json.loads(decoded.decode("utf-8"))
        return True
    except Exception:
        return False


def decode_b64_json(val: str):
    if val is None:
        return ""
    s = str(val).strip()
    if not s:
        return ""
    try:
        decoded = base64.b64decode(s, validate=False)
        obj = json.loads(decoded.decode("utf-8"))
        return json.dumps(obj, ensure_ascii=False, separators=(",", ":"))
    except Exception:
        return s


for csv_path in ALL_FILES:
    if not csv_path.exists():
        print(f"[SKIP: NOT FOUND] {csv_path}")
        continue

    print(f"\n=== Processing {csv_path.name} ===")

    df = pd.read_csv(
        csv_path,
        dtype=str,
        keep_default_na=False,
        na_values=[""],
        engine="python",
    )

    print(f"Loaded {len(df)} rows, {len(df.columns)} columns")

    # --- detect base64 JSON columns ---
    encoded_cols = []
    for col in df.columns:
        sample_vals = df[col].astype(str).str.strip()
        sample_vals = sample_vals[sample_vals != ""]
        if sample_vals.empty:
            continue

        sample = sample_vals.sample(
            n=min(40, len(sample_vals)), random_state=42
        )

        success = 0
        for v in sample:
            if looks_like_base64_json(v):
                success += 1

        if success / len(sample) >= 0.8:
            encoded_cols.append(col)

    if not encoded_cols:
        print("  No base64 JSON columns detected.")
        continue

    print("  Base64 JSON columns:", encoded_cols)

    # --- decode columns ---
    df_dec = df.copy()
    for col in encoded_cols:
        newcol = f"{col}_decoded_json"
        print(f"  Decoding: {col} → {newcol}")
        df_dec[newcol] = df_dec[col].apply(decode_b64_json)

    # --- save output ---
    out_path = csv_path.with_name(csv_path.stem + "_decoded_json.csv")
    df_dec.to_csv(out_path, index=False, encoding="utf-8", lineterminator="\n")

    print(f"  Wrote: {out_path}")



=== Processing battery.csv ===
Loaded 31799 rows, 26 columns
  No base64 JSON columns detected.

=== Processing body.csv ===
Loaded 31799 rows, 12 columns
  No base64 JSON columns detected.

=== Processing engine.csv ===
Loaded 31799 rows, 26 columns
  No base64 JSON columns detected.

=== Processing transmission.csv ===
Loaded 31799 rows, 30 columns
  No base64 JSON columns detected.

=== Processing tyre.csv ===
Loaded 31799 rows, 43 columns
  No base64 JSON columns detected.

=== Processing battery_inference.csv ===
Loaded 31799 rows, 47 columns
  Base64 JSON columns: ['dense_per_feature_error_json', 'explain_top_k_json', 'raw_model_outputs_json']
  Decoding: dense_per_feature_error_json → dense_per_feature_error_json_decoded_json
  Decoding: explain_top_k_json → explain_top_k_json_decoded_json
  Decoding: raw_model_outputs_json → raw_model_outputs_json_decoded_json
  Wrote: C:\Users\ishaa\OneDrive\Desktop\Features-main\Features-main\data\output\battery_inference_decoded_json.csv

=

In [13]:
# Cell 6 - check for JSON cells exceeding Excel's 32,767-character limit in a given CSV

import json
from pathlib import Path
import pandas as pd

# === set the CSV you want to check ===
CHECK_CSV_PATH = Path(r"C:\Users\ishaa\OneDrive\Desktop\Features-main\Features-main\data\output\transmission_inference_decoded_json.csv")

MAX_EXCEL_CHARS = 32767

if not CHECK_CSV_PATH.exists():
    raise FileNotFoundError(f"CSV not found: {CHECK_CSV_PATH}")

print(f"Checking CSV for long JSON cells: {CHECK_CSV_PATH}")

df_check = pd.read_csv(
    CHECK_CSV_PATH,
    dtype=str,
    keep_default_na=False,
    na_values=[""],
    engine="python",
)

print(f"Loaded {len(df_check)} rows, {len(df_check.columns)} columns.")

def looks_like_json(val: str) -> bool:
    if val is None:
        return False
    s = str(val).strip()
    if not s:
        return False
    # quick heuristic: JSON usually starts with { or [
    if not (s.startswith("{") or s.startswith("[")):
        return False
    try:
        json.loads(s)
        return True
    except Exception:
        return False

# Detect JSON-like columns (e.g., *_decoded_json or any col whose values parse as JSON)
json_cols = []
max_samples_per_col = 50

for col in df_check.columns:
    series = df_check[col].astype(str).str.strip()
    non_empty = series[series != ""]
    if non_empty.empty:
        continue

    sample = non_empty.sample(
        n=min(max_samples_per_col, len(non_empty)),
        random_state=42,
    )

    successes = sum(1 for v in sample if looks_like_json(v))
    if successes / len(sample) >= 0.6:  # heuristic: majority look like JSON
        json_cols.append(col)

print("JSON-like columns detected:", json_cols)

any_violations = False
violations_info = []

for col in json_cols:
    lengths = df_check[col].astype(str).str.len()
    mask = lengths > MAX_EXCEL_CHARS
    if mask.any():
        any_violations = True
        violating_indices = mask[mask].index.tolist()
        print(f"\n[VIOLATION] Column '{col}' has {len(violating_indices)} cell(s) > {MAX_EXCEL_CHARS} characters.")
        # store a few examples
        for idx in violating_indices[:5]:
            val = df_check.at[idx, col]
            violations_info.append(
                {
                    "row_index": idx,
                    "column": col,
                    "length": len(str(val)),
                    "preview": str(val)[:120] + ("..." if len(str(val)) > 120 else ""),
                }
            )

if not any_violations:
    print("\nNo JSON-like cells exceed Excel's 32,767-character limit in this CSV.")
else:
    print("\nExamples of offending cells (up to 5):")
    for v in violations_info:
        print(
            f"  Row {v['row_index']} | Col '{v['column']}' | length={v['length']} | preview={v['preview']}"
        )


Checking CSV for long JSON cells: C:\Users\ishaa\OneDrive\Desktop\Features-main\Features-main\data\output\transmission_inference_decoded_json.csv
Loaded 31799 rows, 52 columns.
JSON-like columns detected: ['dense_per_feature_error_json_decoded_json', 'explain_top_k_json_decoded_json', 'raw_model_outputs_json_decoded_json']

No JSON-like cells exceed Excel's 32,767-character limit in this CSV.
